# Satellite Damage Assessment: Model Training

## Notebook 3 - Training Damage Classifiers

This notebook automatically detects whether the data came from **patch mode**
or **scene-level mode** (set in Notebook 2) and trains accordingly:

| Mode | Models trained |
|------|----------------|
| **Patch mode** | ResNet-50 (primary) + Logistic Regression + Random Forest + MLP (baselines) |
| **Scene-level mode** | Logistic Regression + Random Forest + MLP |

All results are saved to `../results/metrics/` for Notebook 4.

## 1. Setup & Load Data

In [ ]:
import pandas as pd
import numpy as np
import os
import json
import pickle
import warnings
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, balanced_accuracy_score
)
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset, WeightedRandomSampler
import torchvision.models as tv_models
import torchvision.transforms.functional as TF

from torchgeo.models import ResNet50_Weights

# Directories
os.makedirs('../models', exist_ok=True)
os.makedirs('../results/metrics', exist_ok=True)
os.makedirs('../results/plots', exist_ok=True)
os.makedirs('../results/predictions', exist_ok=True)

# Device
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'  GPU: {torch.cuda.get_device_name(0)}')
    print(f'  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# Load splits
print('\n' + '='*70)
print('Loading splits from Notebook 2')
print('='*70)

df_train = pd.read_csv('../data/splits/train_metadata.csv')
df_val   = pd.read_csv('../data/splits/val_metadata.csv')
df_test  = pd.read_csv('../data/splits/test_metadata.csv')

print(f'  Train : {len(df_train):,} rows')
print(f'  Val   : {len(df_val):,} rows')
print(f'  Test  : {len(df_test):,} rows')
print(f'  Columns: {df_train.shape[1]}')

# Detect pipeline mode
# Notebook 2 writes a 'temporal_res' column and saves patch .npy files
TEMPORAL_RES = df_train['temporal_res'].iloc[0] if 'temporal_res' in df_train.columns else 'unknown'
PATCH_MODE   = ('patch_id' in df_train.columns and
                os.path.exists('../data/patches/patch_000000.npy'))

print(f'\nDetected mode:')
print(f'  Temporal resolution : {TEMPORAL_RES}')
print(f'  Patch mode          : {PATCH_MODE}')
if PATCH_MODE:
    print('  → Will train ResNet-50 + tabular baselines')
else:
    print('  → Will train tabular baselines only (LR, RF, MLP)')

## 2. Extract Tabular Features & Labels

Used by Logistic Regression, Random Forest, and MLP regardless of mode.

In [ ]:
print('='*70)
print('Extracting Tabular Features & Labels')
print('='*70)

# Feature selection
# Use all numeric columns except metadata and labels.
# Load from saved JSON if it exists (produced by a previous run),
# otherwise derive from the dataframe.
FEATURE_COLS_PATH = '../models/feature_cols.json'
exclude = {
    'patch_id', 'patch_row', 'patch_col',
    'damage_label', 'damage_score',
    'city', 'period', 'year', 'quarter',
    'filename', 'cleaned_filename', '_scene_path', '_split',
    'cloud_cover_pct', 'valid_pixels_pct', 'scene_id', 'temporal_res'
}

feature_cols = [
    c for c in df_train.columns
    if c not in exclude
    and pd.api.types.is_numeric_dtype(df_train[c])
]

print(f'  Selected {len(feature_cols)} feature columns')
print(f'  Features: {feature_cols}')

# Extract arrays
X_train_raw = df_train[feature_cols].values.astype(np.float32)
y_train     = df_train['damage_label'].values.astype(np.int64)

X_val_raw   = df_val[feature_cols].values.astype(np.float32)
y_val       = df_val['damage_label'].values.astype(np.int64)

X_test_raw  = df_test[feature_cols].values.astype(np.float32)
y_test      = df_test['damage_label'].values.astype(np.int64)

# Impute NaNs
nan_count = np.isnan(X_train_raw).sum()
if nan_count > 0:
    print(f'\n  ⚠ {nan_count} NaNs found in training features — imputing with column median')

imputer = SimpleImputer(strategy='median')
X_train_imp = imputer.fit_transform(X_train_raw)
X_val_imp   = imputer.transform(X_val_raw)
X_test_imp  = imputer.transform(X_test_raw)

# Standardize
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imp)
X_val_scaled   = scaler.transform(X_val_imp)
X_test_scaled  = scaler.transform(X_test_imp)

print(f'\n  X_train : {X_train_scaled.shape}  |  y_train : {y_train.shape}')
print(f'  X_val   : {X_val_scaled.shape}  |  y_val   : {y_val.shape}')
print(f'  X_test  : {X_test_scaled.shape}  |  y_test  : {y_test.shape}')
print(f'  NaNs remaining: {np.isnan(X_train_scaled).sum()}')

# Persist preprocessing objects
with open('../models/feature_cols.json', 'w') as f:
    json.dump(feature_cols, f)
with open('../models/imputer.pkl', 'wb') as f:
    pickle.dump(imputer, f)
with open('../models/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print('\n✓ feature_cols.json, imputer.pkl, scaler.pkl saved')

## 3. Train Logistic Regression (Baseline)

In [ ]:
print('='*70)
print('Training Logistic Regression')
print('='*70)

lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_scaled, y_train)

y_train_pred   = lr_model.predict(X_train_scaled)
y_val_pred     = lr_model.predict(X_val_scaled)
y_test_pred    = lr_model.predict(X_test_scaled)
y_test_proba   = lr_model.predict_proba(X_test_scaled)[:, 1]

lr_results = {
    'model'          : 'Logistic Regression',
    'train_acc'      : accuracy_score(y_train, y_train_pred),
    'val_acc'        : accuracy_score(y_val,   y_val_pred),
    'test_acc'       : accuracy_score(y_test,  y_test_pred),
    'test_precision' : precision_score(y_test, y_test_pred, zero_division=0),
    'test_recall'    : recall_score(y_test,    y_test_pred, zero_division=0),
    'test_f1'        : f1_score(y_test,        y_test_pred, zero_division=0),
    'test_auc'       : roc_auc_score(y_test,   y_test_proba)
}

for k, v in lr_results.items():
    if k != 'model':
        print(f'  {k:20s}: {v:.4f}')

with open('../models/logistic_regression.pkl', 'wb') as f:
    pickle.dump(lr_model, f)
print('\n✓ Saved ../models/logistic_regression.pkl')

## 4. Train Random Forest (Baseline)

In [ ]:
print('='*70)
print('Training Random Forest')
print('='*70)

rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train_scaled, y_train)

y_train_pred  = rf_model.predict(X_train_scaled)
y_val_pred    = rf_model.predict(X_val_scaled)
y_test_pred   = rf_model.predict(X_test_scaled)
y_test_proba  = rf_model.predict_proba(X_test_scaled)[:, 1]

rf_results = {
    'model'          : 'Random Forest',
    'train_acc'      : accuracy_score(y_train, y_train_pred),
    'val_acc'        : accuracy_score(y_val,   y_val_pred),
    'test_acc'       : accuracy_score(y_test,  y_test_pred),
    'test_precision' : precision_score(y_test, y_test_pred, zero_division=0),
    'test_recall'    : recall_score(y_test,    y_test_pred, zero_division=0),
    'test_f1'        : f1_score(y_test,        y_test_pred, zero_division=0),
    'test_auc'       : roc_auc_score(y_test,   y_test_proba)
}

for k, v in rf_results.items():
    if k != 'model':
        print(f'  {k:20s}: {v:.4f}')

feature_importance = pd.DataFrame({
    'feature'   : feature_cols,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

print(f'\nTop 10 Features:')
print(feature_importance.head(10).to_string(index=False))

with open('../models/random_forest.pkl', 'wb') as f:
    pickle.dump(rf_model, f)
print('\n✓ Saved ../models/random_forest.pkl')

## 5. Train MLP (Baseline)

In [ ]:
print('='*70)
print('Training MLP Classifier')
print('='*70)

class TabularMLP(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, 64), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(64, 32),         nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(32, 2)
        )
    def forward(self, x):
        return self.net(x)

X_train_t = torch.FloatTensor(X_train_scaled).to(DEVICE)
y_train_t = torch.LongTensor(y_train).to(DEVICE)
X_val_t   = torch.FloatTensor(X_val_scaled).to(DEVICE)
y_val_t   = torch.LongTensor(y_val).to(DEVICE)
X_test_t  = torch.FloatTensor(X_test_scaled).to(DEVICE)

train_loader_tab = DataLoader(
    TensorDataset(X_train_t, y_train_t), batch_size=32, shuffle=True
)

nn_model  = TabularMLP(input_size=X_train_scaled.shape[1]).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(nn_model.parameters(), lr=1e-3)

MLP_EPOCHS = 50
best_val_loss = float('inf')
mlp_train_losses, mlp_val_losses = [], []

print(f'Training for {MLP_EPOCHS} epochs...')
for epoch in range(MLP_EPOCHS):
    nn_model.train()
    ep_loss = 0.0
    for xb, yb in train_loader_tab:
        optimizer.zero_grad()
        loss = criterion(nn_model(xb), yb)
        loss.backward()
        optimizer.step()
        ep_loss += loss.item()
    ep_loss /= len(train_loader_tab)
    mlp_train_losses.append(ep_loss)

    nn_model.eval()
    with torch.no_grad():
        vl = criterion(nn_model(X_val_t), y_val_t).item()
    mlp_val_losses.append(vl)

    if vl < best_val_loss:
        best_val_loss = vl
        torch.save(nn_model.state_dict(), '../models/mlp_best.pth')

    if (epoch + 1) % 10 == 0:
        print(f'  Epoch {epoch+1:3d}/{MLP_EPOCHS}  train={ep_loss:.4f}  val={vl:.4f}')

# Load best weights and evaluate
nn_model.load_state_dict(torch.load('../models/mlp_best.pth', map_location=DEVICE))
nn_model.eval()
with torch.no_grad():
    y_train_pred  = torch.argmax(nn_model(X_train_t), 1).cpu().numpy()
    y_val_pred    = torch.argmax(nn_model(X_val_t),   1).cpu().numpy()
    test_out      = nn_model(X_test_t)
    y_test_pred   = torch.argmax(test_out, 1).cpu().numpy()
    y_test_proba  = torch.softmax(test_out, 1)[:, 1].cpu().numpy()

nn_results = {
    'model'          : 'MLP',
    'train_acc'      : accuracy_score(y_train, y_train_pred),
    'val_acc'        : accuracy_score(y_val,   y_val_pred),
    'test_acc'       : accuracy_score(y_test,  y_test_pred),
    'test_precision' : precision_score(y_test, y_test_pred, zero_division=0),
    'test_recall'    : recall_score(y_test,    y_test_pred, zero_division=0),
    'test_f1'        : f1_score(y_test,        y_test_pred, zero_division=0),
    'test_auc'       : roc_auc_score(y_test,   y_test_proba)
}

for k, v in nn_results.items():
    if k != 'model':
        print(f'  {k:20s}: {v:.4f}')

# Save final weights (the best already saved above)
torch.save(nn_model.state_dict(), '../models/neural_network.pth')
print('\n✓ Saved ../models/neural_network.pth')

In [ ]:
# Check if patches are valid 
import random

print("DIAGNOSTIC: Checking patch files...")
print("=" * 70)

# Check a random patch
sample_idx = random.randint(0, len(df_train) - 1)
sample_row = df_train.iloc[sample_idx]
sample_patch_id = int(sample_row['patch_id'])
sample_path = f'../data/patches/patch_{sample_patch_id:06d}.npy'

print(f"Loading sample patch: {sample_path}")
sample_patch = np.load(sample_path)

print(f"  Shape: {sample_patch.shape}")
print(f"  Dtype: {sample_patch.dtype}")
print(f"  Min value: {np.nanmin(sample_patch):.2f}")
print(f"  Max value: {np.nanmax(sample_patch):.2f}")
print(f"  Mean value: {np.nanmean(sample_patch):.2f}")
print(f"  Contains NaN: {np.isnan(sample_patch).any()}")
print(f"  Contains Inf: {np.isinf(sample_patch).any()}")
print(f"  All zeros: {(sample_patch == 0).all()}")

print(f"\nChecking all patches...")
has_nan = 0
has_inf = 0
all_zeros = 0

for i in range(min(100, len(df_train))):
    patch_id = int(df_train.iloc[i]['patch_id'])
    path = f'../data/patches/patch_{patch_id:06d}.npy'
    try:
        p = np.load(path)
        if np.isnan(p).any():
            has_nan += 1
        if np.isinf(p).any():
            has_inf += 1
        if (p == 0).all():
            all_zeros += 1
    except Exception as e:
        print(f"Error loading patch {patch_id}: {e}")

print(f"  NaN found in {has_nan}/100 patches")
print(f"  Inf found in {has_inf}/100 patches")
print(f"  All zeros: {all_zeros}/100 patches")

if has_nan > 0 or has_inf > 0 or all_zeros > 10:
    print("\n  PROBLEM: Patches are corrupted!")
else:
    print("\n✓ Patches look OK")

## 6. ResNet-50 Training (Patch Mode Only)

This section runs **only when patch `.npy` files are present**.
If you are in scene-level mode it is skipped automatically.

In [ ]:
resnet_results = None  # will remain None if patch mode is off
resnet_train_losses, resnet_val_losses = [], []

if not PATCH_MODE:
    print('Scene-level mode detected — skipping ResNet-50.')
    print('Set USE_PATCHES=True in Notebook 2 and re-run to enable ResNet-50.')
else:
    print('='*70)
    print('ResNet-50 Patch Dataset & DataLoader')
    print('='*70)

    # Dataset
    class SentinelPatchDataset(Dataset):
        """
        Loads 128x128 Sentinel-2 patches from .npy files.
        Each patch has shape (H, W, C) — we convert to (C, H, W) for PyTorch.
        We use the first 6 bands (Blue, Green, Red, NIR, SWIR1, SWIR2).
        Pixels are normalised to [0, 1] by dividing by 10000 (Sentinel-2 scale).
        """
        def __init__(self, df, patch_dir='../data/patches', n_bands=6, augment=False):
            self.df        = df.reset_index(drop=True)
            self.patch_dir = patch_dir
            self.n_bands   = n_bands
            self.augment   = augment

        def __len__(self):
            return len(self.df)

        def __getitem__(self, idx):
            row      = self.df.iloc[idx]
            patch_id = int(row['patch_id'])
            label    = int(row['damage_label'])

            path  = os.path.join(self.patch_dir, f'patch_{patch_id:06d}.npy')
            patch = np.load(path).astype(np.float32)          # (H, W, C)

            # Keep only the first n_bands
            if patch.ndim == 3 and patch.shape[2] >= self.n_bands:
                patch = patch[:, :, :self.n_bands]
            elif patch.ndim == 3 and patch.shape[2] < self.n_bands:
                # Pad missing bands with zeros
                pad   = np.zeros((*patch.shape[:2], self.n_bands - patch.shape[2]), dtype=np.float32)
                patch = np.concatenate([patch, pad], axis=2)

            # Normalise: Sentinel-2 surface reflectance values are 0–10000
            patch = np.clip(patch / 10000.0, 0.0, 1.0)

            # (H, W, C) → (C, H, W)
            tensor = torch.from_numpy(patch.transpose(2, 0, 1))
            
            # ImageNet normalization for RGB bands (first 3 channels)
            imagenet_mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
            imagenet_std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

            # Apply to first 3 bands only (Blue, Green, Red)
            tensor[:3, :, :] = (tensor[:3, :, :] - imagenet_mean) / imagenet_std

            # For NIR, SWIR1, SWIR2 (bands 4-6), use simple [0,1] as-is
            # They don't have ImageNet priors, so [0,1] is reasonable

            # Light augmentation on training patches
            if self.augment:
                if torch.rand(1).item() > 0.5:
                    tensor = TF.hflip(tensor)
                if torch.rand(1).item() > 0.5:
                    tensor = TF.vflip(tensor)

            return tensor, label

    BATCH_SIZE = 32
    N_BANDS    = 6

    # Class weights to handle potential imbalance
    class_counts = np.bincount(y_train)
    # class_weights = torch.FloatTensor(1.0 / class_counts).to(DEVICE)
    # class_weights = class_weights / class_weights.sum()
    weight_for_0 = len(y_train) / (2 * class_counts[0])  # undamaged (minority)
    weight_for_1 = len(y_train) / (2 * class_counts[1])  # damaged (majority)
    class_weights = torch.FloatTensor([weight_for_0, weight_for_1]).to(DEVICE)

    sample_weights = np.where(y_train == 0, weight_for_0, weight_for_1)
    sampler = WeightedRandomSampler(
        weights=torch.FloatTensor(sample_weights),
        num_samples=len(y_train),
        replacement=True
    )

    train_ds = SentinelPatchDataset(df_train, augment=True)
    val_ds   = SentinelPatchDataset(df_val,   augment=False)
    test_ds  = SentinelPatchDataset(df_test,  augment=False)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                            num_workers=0, pin_memory=(DEVICE.type == 'cuda'))
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=0, pin_memory=(DEVICE.type == 'cuda'))
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=0, pin_memory=(DEVICE.type == 'cuda'))

    print(f'  Train batches : {len(train_loader)}')
    print(f'  Val batches   : {len(val_loader)}')
    print(f'  Test batches  : {len(test_loader)}')

    # Model
    print('\nBuilding ResNet-50...')
    
    resnet = tv_models.resnet50(weights=tv_models.ResNet50_Weights.IMAGENET1K_V2)
    resnet = tv_models.resnet50(weights=None)
    weights = ResNet50_Weights.SENTINEL2_ALL_MOCO
    # Replace first conv layer: 3 RGB channels → N_BANDS Sentinel-2 channels
    # We copy the pretrained RGB weights into the first 3 channels and
    # initialise the extra channels as the mean of those 3 (a sensible prior).
    old_conv   = resnet.conv1
    new_conv   = nn.Conv2d(
        N_BANDS, old_conv.out_channels,
        kernel_size=old_conv.kernel_size,
        stride=old_conv.stride,
        padding=old_conv.padding,
        bias=False
    )
    with torch.no_grad():
        new_conv.weight[:, :3, :, :] = old_conv.weight           # copy RGB weights
        for b in range(3, N_BANDS):                               # initialise extra bands
            new_conv.weight[:, b, :, :] = old_conv.weight.mean(dim=1)
    resnet.conv1 = new_conv

    # Replace final fully-connected layer for binary classification
    in_features   = resnet.fc.in_features
    resnet.fc     = nn.Sequential(
        nn.Dropout(0.5),
        nn.Linear(in_features, 2)
    )

    resnet = resnet.to(DEVICE)
    total_params     = sum(p.numel() for p in resnet.parameters())
    trainable_params = sum(p.numel() for p in resnet.parameters() if p.requires_grad)
    print(f'  Total params     : {total_params:,}')
    print(f'  Trainable params : {trainable_params:,}')
    print(f'  Input channels   : {N_BANDS}')

    # Training
    RESNET_EPOCHS = 40
    LR_INIT       = 1e-4
    PATIENCE      = 7     # early stopping patience

    resnet_criterion = nn.CrossEntropyLoss(weight=class_weights)
    resnet_optimizer = optim.AdamW(resnet.parameters(), lr=LR_INIT, weight_decay=1e-4)

    # Cosine annealing LR schedule
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        resnet_optimizer, T_max=RESNET_EPOCHS, eta_min=1e-6
    )

    print(f'\nTraining ResNet-50 for up to {RESNET_EPOCHS} epochs '
          f'(early stopping patience={PATIENCE})...')
    print(f'  LR={LR_INIT}, weight_decay=1e-4, cosine annealing')
    print(f'  Class weights: {class_weights.cpu().numpy().round(4)}')
    print()

    best_resnet_val_loss = 0.0  # now tracking best balanced accuracy
    patience_counter     = 0

    for epoch in range(RESNET_EPOCHS):
        # ── Train ──
        resnet.train()
        ep_loss, ep_correct, ep_total = 0.0, 0, 0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            resnet_optimizer.zero_grad()
            out  = resnet(imgs)
            loss = resnet_criterion(out, labels)
            loss.backward()
            resnet_optimizer.step()
            ep_loss    += loss.item() * imgs.size(0)
            ep_correct += (out.argmax(1) == labels).sum().item()
            ep_total   += imgs.size(0)

        train_loss = ep_loss / ep_total
        train_acc  = ep_correct / ep_total
        resnet_train_losses.append(train_loss)

        # Validate
        vl_loss, vl_correct, vl_total = 0.0, 0, 0
        val_preds_ep = []
        val_labels_ep = []
        resnet.eval()
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                out = resnet(imgs)
                val_preds_ep.extend(out.argmax(1).cpu().numpy())
                val_labels_ep.extend(labels.cpu().numpy())
                vl_loss    += resnet_criterion(out, labels).item() * imgs.size(0)
                vl_correct += (out.argmax(1) == labels).sum().item()
                vl_total   += imgs.size(0)

        val_loss     = vl_loss / vl_total
        val_acc      = vl_correct / vl_total
        val_bal_acc  = balanced_accuracy_score(val_labels_ep, val_preds_ep)  # ← use this
        resnet_val_losses.append(val_loss)

        print(f'  Epoch {epoch+1:3d}/{RESNET_EPOCHS}  '
            f'train_loss={train_loss:.4f}  train_acc={train_acc:.4f}  '
            f'val_loss={val_loss:.4f}  val_acc={val_acc:.4f}  '
            f'val_bal_acc={val_bal_acc:.4f}  lr={scheduler.get_last_lr()[0]:.2e}')

        # Early stopping on balanced accuracy (higher is better)
        if val_bal_acc > best_resnet_val_loss:   # reuse variable, now tracking best bal_acc
            best_resnet_val_loss = val_bal_acc
            patience_counter = 0
            torch.save(resnet.state_dict(), '../models/resnet50_best.pth')
        else:
            patience_counter += 1

    # Evaluate on test set
    print('\nLoading best ResNet-50 checkpoint for test evaluation...')
    resnet.load_state_dict(torch.load('../models/resnet50_best.pth', map_location=DEVICE))
    resnet.eval()

    all_preds, all_proba, all_labels = [], [], []
    with torch.no_grad():
        for imgs, labels in test_loader:
            imgs = imgs.to(DEVICE)
            out  = resnet(imgs)
            prob = torch.softmax(out, dim=1)[:, 1].cpu().numpy()
            pred = out.argmax(1).cpu().numpy()
            all_preds.append(pred)
            all_proba.append(prob)
            all_labels.append(labels.numpy())

    y_test_pred_resnet  = np.concatenate(all_preds)
    y_test_proba_resnet = np.concatenate(all_proba)
    y_test_labels       = np.concatenate(all_labels)

    # Also get train/val acc for the comparison table
    def eval_loader(loader, model):
        correct, total = 0, 0
        model.eval()
        with torch.no_grad():
            for imgs, labels in loader:
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                correct += (model(imgs).argmax(1) == labels).sum().item()
                total   += imgs.size(0)
        return correct / total

    resnet_results = {
        'model'          : 'ResNet-50',
        'train_acc'      : eval_loader(train_loader, resnet),
        'val_acc'        : eval_loader(val_loader,   resnet),
        'test_acc'       : accuracy_score(y_test_labels,  y_test_pred_resnet),
        'test_precision' : precision_score(y_test_labels, y_test_pred_resnet, zero_division=0),
        'test_recall'    : recall_score(y_test_labels,    y_test_pred_resnet, zero_division=0),
        'test_f1'        : f1_score(y_test_labels,        y_test_pred_resnet, zero_division=0),
        'test_auc'       : roc_auc_score(y_test_labels,   y_test_proba_resnet)
    }

    print('\nResNet-50 Test Results:')
    for k, v in resnet_results.items():
        if k != 'model':
            print(f'  {k:20s}: {v:.4f}')

    print('\n✓ Best weights saved to ../models/resnet50_best.pth')

## 7. Model Comparison

In [ ]:
print('='*70)
print('MODEL COMPARISON')
print('='*70)

all_results = [lr_results, rf_results, nn_results]
if resnet_results is not None:
    all_results.append(resnet_results)

results_df = pd.DataFrame(all_results)
print(results_df.to_string(index=False))

best_idx        = results_df['test_f1'].idxmax()
best_model_name = results_df.loc[best_idx, 'model']
best_f1         = results_df.loc[best_idx, 'test_f1']

print(f'\nBest model: {best_model_name}  (F1 = {best_f1:.4f})')

results_df.to_csv('../results/metrics/model_comparison.csv', index=False)
print('\n✓ Saved ../results/metrics/model_comparison.csv')

## 8. Visualizations

In [ ]:
n_models = len(results_df)
colors   = ['steelblue', 'seagreen', 'darkorange', 'crimson'][:n_models]
model_names = results_df['model'].values

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle(
    f'Model Comparison — {TEMPORAL_RES.capitalize()} / {"Patch" if PATCH_MODE else "Scene-level"} mode',
    fontsize=14, fontweight='bold'
)

# 1. Accuracy
axes[0,0].bar(model_names, results_df['test_acc'], color=colors, alpha=0.85)
axes[0,0].set_title('Test Accuracy'); axes[0,0].set_ylim(0, 1)
axes[0,0].grid(True, alpha=0.3, axis='y')

# 2. F1 Score
axes[0,1].bar(model_names, results_df['test_f1'], color=colors, alpha=0.85)
axes[0,1].set_title('Test F1 Score'); axes[0,1].set_ylim(0, 1)
axes[0,1].grid(True, alpha=0.3, axis='y')

# 3. AUC
axes[0,2].bar(model_names, results_df['test_auc'], color=colors, alpha=0.85)
axes[0,2].set_title('Test AUC'); axes[0,2].set_ylim(0, 1)
axes[0,2].grid(True, alpha=0.3, axis='y')

# 4. MLP training curves
axes[1,0].plot(mlp_train_losses, label='Train', linewidth=2)
axes[1,0].plot(mlp_val_losses,   label='Val',   linewidth=2)
axes[1,0].set_title('MLP Training Curves')
axes[1,0].set_xlabel('Epoch'); axes[1,0].set_ylabel('Loss')
axes[1,0].legend(); axes[1,0].grid(True, alpha=0.3)

# 5. ResNet-50 training curves (or feature importance if no patches)
if PATCH_MODE and resnet_train_losses:
    axes[1,1].plot(resnet_train_losses, label='Train', linewidth=2, color='crimson')
    axes[1,1].plot(resnet_val_losses,   label='Val',   linewidth=2, color='salmon')
    axes[1,1].set_title('ResNet-50 Training Curves')
    axes[1,1].set_xlabel('Epoch'); axes[1,1].set_ylabel('Loss')
    axes[1,1].legend(); axes[1,1].grid(True, alpha=0.3)
else:
    top10 = feature_importance.head(10)
    axes[1,1].barh(range(len(top10)), top10['importance'].values, color='seagreen', alpha=0.8)
    axes[1,1].set_yticks(range(len(top10)))
    axes[1,1].set_yticklabels(top10['feature'].values, fontsize=8)
    axes[1,1].invert_yaxis()
    axes[1,1].set_title('Top 10 Features (RF)')
    axes[1,1].grid(True, alpha=0.3, axis='x')

# 6. Confusion matrix for best model
if best_model_name == 'Logistic Regression':
    y_pred_best = lr_model.predict(X_test_scaled)
elif best_model_name == 'Random Forest':
    y_pred_best = rf_model.predict(X_test_scaled)
elif best_model_name == 'ResNet-50':
    y_pred_best = y_test_pred_resnet
    y_test      = y_test_labels
else:
    with torch.no_grad():
        y_pred_best = torch.argmax(nn_model(X_test_t), 1).cpu().numpy()

cm = confusion_matrix(y_test, y_pred_best)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1,2],
            xticklabels=['Not Damaged', 'Damaged'],
            yticklabels=['Not Damaged', 'Damaged'])
axes[1,2].set_title(f'Confusion Matrix\n({best_model_name})')
axes[1,2].set_ylabel('True'); axes[1,2].set_xlabel('Predicted')

plt.tight_layout()
plt.savefig('../results/plots/model_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Saved ../results/plots/model_evaluation.png')

# Feature importance (separate plot, always useful)
top20 = feature_importance.head(20)
fig2, ax2 = plt.subplots(figsize=(10, 7))
ax2.barh(range(len(top20)), top20['importance'].values, color='seagreen', alpha=0.85)
ax2.set_yticks(range(len(top20)))
ax2.set_yticklabels(top20['feature'].values, fontsize=9)
ax2.invert_yaxis()
ax2.set_xlabel('Importance')
ax2.set_title('Top 20 Features — Random Forest', fontweight='bold')
ax2.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig('../results/plots/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Saved ../results/plots/feature_importance.png')

## 9. Save Predictions for Notebook 4

In [ ]:
print('='*70)
print('Saving Test Predictions for Notebook 4')
print('='*70)

# Use best model to generate probabilities
if best_model_name == 'Logistic Regression':
    proba_best = lr_model.predict_proba(X_test_scaled)[:, 1]
    pred_best  = lr_model.predict(X_test_scaled)
elif best_model_name == 'Random Forest':
    proba_best = rf_model.predict_proba(X_test_scaled)[:, 1]
    pred_best  = rf_model.predict(X_test_scaled)
elif best_model_name == 'ResNet-50':
    proba_best = y_test_proba_resnet
    pred_best  = y_test_pred_resnet
else:
    with torch.no_grad():
        out        = nn_model(X_test_t)
        proba_best = torch.softmax(out, 1)[:, 1].cpu().numpy()
        pred_best  = out.argmax(1).cpu().numpy()

predictions = df_test.copy()
predictions['predicted_damage']   = pred_best
predictions['damage_probability']  = proba_best
predictions['correct_prediction']  = (predictions['damage_label'] == predictions['predicted_damage']).astype(int)
predictions['best_model']          = best_model_name

predictions.to_csv('../results/predictions/test_predictions.csv', index=False)

print(f'  Correct: {predictions["correct_prediction"].sum()}/{len(predictions)}')
print(f'  Accuracy: {predictions["correct_prediction"].mean():.4f}')
print(f'\n  Sample predictions:')
cols = ['city', 'period', 'damage_label', 'predicted_damage', 'damage_probability', 'correct_prediction']
cols = [c for c in cols if c in predictions.columns]
print(predictions[cols].head(10).to_string(index=False))
print('\n✓ Saved ../results/predictions/test_predictions.csv')

## 10. Final Summary

In [ ]:
print('='*70)
print('MODEL TRAINING COMPLETE')
print('='*70)
print(f'  Mode              : {TEMPORAL_RES} / {"patch" if PATCH_MODE else "scene-level"}')
print(f'  Best model        : {best_model_name}')
print(f'  Test F1           : {best_f1:.4f}')
print(f'  Test Accuracy     : {results_df.loc[best_idx, "test_acc"]:.4f}')
print(f'  Test AUC          : {results_df.loc[best_idx, "test_auc"]:.4f}')
print()
print('Saved files:')
print('  ../models/logistic_regression.pkl')
print('  ../models/random_forest.pkl')
print('  ../models/neural_network.pth  (MLP best weights)')
if PATCH_MODE:
    print('  ../models/resnet50_best.pth')
print('  ../models/feature_cols.json')
print('  ../models/imputer.pkl')
print('  ../models/scaler.pkl')
print('  ../results/metrics/model_comparison.csv')
print('  ../results/predictions/test_predictions.csv')
print('  ../results/plots/model_evaluation.png')
print('  ../results/plots/feature_importance.png')
print()
print('→ Run Notebook 4 for full evaluation and damage maps')